<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_42_api_testing/note_lesson_42_testing_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Урок 42 — Тестування API: новинний агрегатор, крок 5

`news_hub` уже збирає новини з rbc.ua, зберігає їх у базу, кешує в Redis і обмежує частоту запитів. Тести в нього були — і всі зелені. Сьогодні перебудовуємо їх у **шари** і з'ясовуємо, чому зелені тести пропустили чотири справжні помилки: час з атрибута `<time>`, битий байт на сторінці, `fakerbc.ua` і стару версію бібліотеки.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | `news_hub` і pytest з ноутбука | 0.1 | |
| 1 | шари тестів: unit і integration, маркери, вибір тестів | 1.1–1.4 | |
| 2 | збережені сторінки, фабрика даних, тест конвеєра | 2.1–2.5 | |
| 3 | мок мережі: `MagicMock`, помилки мережі, **patch where used**, шпигун `wraps` | 3.1–3.5 | **Задача 1** |
| 4 | фейк замість моку: локальний HTTP-сервер; межа фейку — fakeredis | 4.1–4.4 | **Задача 2** |
| 5 | API асинхронно: `httpx.AsyncClient`, тестова база, кеш, rate limit | 5.1–5.4 | |
| 6 | покриття, справжній `get_db`, тест ключа кешу і чи він ловить помилку | 6.1–6.5 | |
| 7 | межові значення: «майже rbc.ua»; мінімальні версії | 7.1–7.3 | **Задача 3** |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, `class`, константи) лише створює їх — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Серверів і інтернету не треба: база — SQLite, Redis — `fakeredis`, «сайт» — локальний `aiohttp`-сервер на `127.0.0.1`.
- Час виконання у тебе буде інший — важливі статуси, кількості й висновки.
- Теорія й розбір знахідок — на [сторінці уроку](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_42/).

---
## 0. Проєкт і pytest

Клітинка завантажує `news_hub` (у Colab), ставить залежності й переходить у папку проєкту. Змінні середовища кажуть застосунку: база — файл SQLite, Redis — `fakeredis` у пам'яті.

У ноутбуці можна писати `await` прямо в клітинці: Jupyter уже має запущений цикл подій (тому `asyncio.run(...)` тут не працює).

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_42_api_testing"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("news_hub"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import aiosqlite, fakeredis, fastapi, httpx, pytest_asyncio, pytest_cov  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "news_hub/requirements.txt"], check=True)

os.chdir("news_hub")
sys.path.insert(0, os.getcwd())
os.environ["DATABASE_URL"] = "sqlite+aiosqlite:///notebook.db"
os.environ["REDIS_URL"] = "fakeredis://"
print("папка проєкту:", os.getcwd())

`run_pytest(...)` запускає pytest **окремим процесом** — так само, як у терміналі — і повертає вивід рядком. Окремий процес важливий: тести самі налаштовують змінні середовища й підміняють залежності застосунку, і це не повинно зачепити ноутбук.

In [ ]:
def run_pytest(*args: str) -> str:
    """pytest окремим процесом (як у терміналі); повертає вивід."""
    env = {key: value for key, value in os.environ.items() if key not in ("DATABASE_URL", "REDIS_URL")}
    result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no", *args],
                            capture_output=True, text=True, env=env)
    return result.stdout + result.stderr

### Приклад 0.1. Усі тести проєкту

In [ ]:
print(run_pytest().splitlines()[-1])

Один рядок — підсумок: скільки пройшло і за скільки секунд. `-q` — короткий вивід, `-p no:cacheprovider` — не створювати папку `.pytest_cache` у проєкті.

---
## 1. Шари тестів

`tests/unit/` — одна функція чи клас **без** бази, Redis і мережі. `tests/integration/` — застосунок цілком.

```text
tests/
├── conftest.py        ← фікстура html(), маркер за папкою
├── factories.py       ← make_raw(**зміни)
├── fixtures/          ← збережені сторінки *.html
├── unit/              ← test_parser, test_models, test_pipeline, test_scraper
└── integration/       ← conftest (client, aclient, тестова база), test_api, test_crud, test_redis,
                         test_async_api, test_db, test_scraper_server
```

### Приклад 1.1. Маркер ставить сама папка

`pytest_collection_modifyitems` — **хук** pytest: функція з цим ім'ям у `conftest.py` отримує всі зібрані тести до запуску. Тут вона додає маркер `unit` або `integration` за назвою папки:

In [ ]:
import inspect

from tests import conftest

print(inspect.getsource(conftest.pytest_collection_modifyitems))
print(open("pytest.ini", encoding="utf-8").read())

Маркери оголошено в `pytest.ini` — інакше pytest попередить про невідомий маркер. `pythonpath = .` дає тестам імпортувати `news_hub`.

### Приклад 1.2. Лише unit або лише integration

`-m unit` вибирає тести з маркером, решту pytest позначає як `deselected`:

In [ ]:
unit_line = run_pytest("-m", "unit").splitlines()[-1]
integration_line = run_pytest("-m", "integration").splitlines()[-1]
print("unit:       ", unit_line)
print("integration:", integration_line)
assert "passed" in unit_line and "deselected" in unit_line
assert "passed" in integration_line and "failed" not in integration_line

Unit-тести йдуть близько секунди — їх ганяють після кожної зміни. Повний набір — перед комітом і в CI (урок 51).

### Приклад 1.3. Скільки тестів у кожному файлі

`--collect-only` лише **збирає** тести, не запускаючи. Кожен рядок `-q`-виводу — ідентифікатор тесту `файл::функція[параметри]`. Порахуємо їх за файлами:

In [ ]:
from collections import Counter


def tests_per_file(*args: str) -> Counter:
    """Скільки тестів pytest збирає з кожного файлу."""
    ids = [line for line in run_pytest("--collect-only", *args).splitlines() if "::" in line]
    return Counter(test_id.split("::")[0] for test_id in ids)

In [ ]:
for path, count in sorted(tests_per_file().items()):
    print(f"{path:45} {count:3}")

Один `parametrize` дає кілька тестів: у `test_models.py` їх більше, ніж функцій `def test_…`.

### Приклад 1.4. Один тест за ім'ям: `-k` і `-v`

`-k вираз` — тести, в імені яких є підрядок; `-v` — показати кожен тест окремо.

In [ ]:
verbose = run_pytest("-k", "rbc_host or patch_where_used", "-v", "-v")   # -q у run_pytest гасить один -v
print("\n".join(line.split(" PASSED")[0] for line in verbose.splitlines() if " PASSED" in line))
print(verbose.splitlines()[-1])

Ідентифікатори параметризованого тесту містять параметри у квадратних дужках — `[fakerbc.ua-False]`: одразу видно, **який** випадок упав.

---
## 2. Збережені сторінки

Тести парсера раніше містили HTML прямо в коді — рядки, які автор тесту придумав сам. Тепер розмітка лежить у файлах `tests/fixtures/`:

- `rbc_newsline_item.html` — **справжній** фрагмент стрічки rbc.ua: `div.item > a > span.time`, контейнерів `newsline__item` немає;
- `demo_newsline.html` — демо-розмітка: контейнери `newsline__item`, час в атрибуті `<time datetime="…">`.

### Приклад 2.1. Справжній фрагмент стрічки → парсер

In [ ]:
from pathlib import Path

from news_hub.models import NewsItem, validate_news
from news_hub.parser import parse_rbc_news

FIXTURES = Path("tests/fixtures")

In [ ]:
news = parse_rbc_news((FIXTURES / "rbc_newsline_item.html").read_text(encoding="utf-8"))
print(news)
assert len(news) == 1 and news[0]["datetime"] == "02:06"

Одна новина: заголовок, адреса й час «02:06» з `span.time`. Коли сайт змінить розмітку, файл оновлюють збереженою сторінкою — і тести покажуть, що зламалось.

### Приклад 2.2. Фікстура `html(name)`

У тестах сторінку бере фікстура з `tests/conftest.py`: `html("demo_newsline")` → вміст файлу. Фікстура **повертає функцію** — так один тест може прочитати кілька сторінок:

In [ ]:
print(inspect.getsource(conftest.html))

Викликати фікстуру напряму pytest не дозволяє (параметри їй передає сам pytest за іменем), тож у ноутбуці визначимо ту саму функцію:

In [ ]:
def html(name: str) -> str:
    """html("demo_newsline") → вміст tests/fixtures/demo_newsline.html."""
    return (FIXTURES / f"{name}.html").read_text(encoding="utf-8")

In [ ]:
demo = html("demo_newsline")
print(len(demo), "символів |", demo.count("newsline__item"), "згадок newsline__item")
print([item["datetime"] for item in parse_rbc_news(demo)])

Зверни увагу на час: `"2024-05-08T10:30:00"` — **ISO-рядок** з атрибута, а не «10:30».

### Приклад 2.3. Фабрика: у тесті видно лише важливе поле

`make_raw(**зміни)` з `tests/factories.py` — сира новина з правильними значеннями, в якій змінено лише те, що перевіряє тест:

In [ ]:
from tests.factories import make_raw

print(make_raw())
print(make_raw(title="Коротко"))

In [ ]:
valid, rejected = validate_news([make_raw(), make_raw(title="Коротко"), make_raw(url="https://fakerbc.ua/ukr/news/1.html")])
print("прийнято:", len(valid))
for item in rejected:
    print("відхилено:", item.errors)

`validate_news` нічого не губить мовчки: погані новини — у `rejected` з причиною (заголовок коротший за 10 символів; адреса не з rbc.ua).

### Приклад 2.4. Знайди помилку: два зелені тести

Ось два тести з попередніх уроків (скорочено) і модель тих уроків — лише поле часу. Кожен тест правильний і зелений:

In [ ]:
from datetime import time

from pydantic import BaseModel, ValidationError


class NewsTimeOld(BaseModel):                 # поле published_time моделі попередніх уроків
    published_time: time | None = None


def test_parser_reads_time_attribute() -> None:
    page = ('<div class="newsline__item"><a class="title" href="/ukr/news/1/">Уряд затвердив новий бюджет</a>'
            '<time datetime="2024-05-08T10:30:00">10:30</time></div>')
    assert parse_rbc_news(page)[0]["datetime"] == "2024-05-08T10:30:00"


def test_model_parses_time() -> None:
    assert NewsTimeOld(published_time="14:19").published_time == time(14, 19)

In [ ]:
test_parser_reads_time_attribute()
test_model_parses_time()
print("обидва тести зелені")

for item in parse_rbc_news(demo):
    try:
        NewsTimeOld(published_time=item["datetime"])
    except ValidationError as error:
        print(f"{item['datetime']!r} → {error.errors()[0]['msg']}")

**🔮 Прогноз:** кожен тест правильний. Чому разом вони пропустили помилку, і який тест її ловить?

<details>
<summary>Відповідь</summary>

Кожен тест перевіряє свою половину **на своїх даних**: тест парсера — що він віддає ISO-рядок, тест моделі — що вона приймає «14:19». Ніхто не перевіряв **контракт між ними**: чи приймає модель те, що віддає парсер. Ловить **тест конвеєра** — наступний приклад. Правильне рішення — `field_validator("published_time", mode="before")` у `NewsItem`, що бере час з ISO-рядка.

</details>

### Приклад 2.5. Тест конвеєра

Сторінка → `parse_rbc_news` → `validate_news`: усе, що дає парсер на збереженій сторінці, модель має прийняти. Так написано `tests/unit/test_pipeline.py` — `parametrize` за двома сторінками:

In [ ]:
PIPELINE_CASES = {
    "rbc_newsline_item": [("США хочуть підкупити кубинців безплатним інтернетом, - AP", "ru", "02:06:00")],
    "demo_newsline": [("Уряд затвердив новий бюджет на 2024 рік", "uk", "10:30:00"),
                      ("Збірна України перемогла у фіналі", "uk", "09:15:00")],
}


def pipeline(page: str) -> tuple[list[tuple[str, str, str]], int]:
    """(заголовок, мова, час) прийнятих новин і кількість відхилених."""
    valid, rejected = validate_news(parse_rbc_news(html(page)))
    return [(item.title, item.lang, item.published_time.isoformat()) for item in valid], len(rejected)

In [ ]:
for page, expected in PIPELINE_CASES.items():
    got, rejected_count = pipeline(page)
    print(f"{page:18} відхилено: {rejected_count} | {got}")
    assert (got, rejected_count) == (expected, 0)

Нуль відхилених на обох сторінках, а час з ISO-рядка став `10:30:00`. Правило: коли дві частини з'єднані даними, потрібен хоча б один тест, у якому дані з однієї **справді** йдуть у другу.

---
## 3. Мок мережі

`fetch_one` робить HTTP-запит. У тесті інтернету може не бути, а rbc.ua може відповісти 403 — тест залежав би від погоди. **Мок** замінює `aiohttp.ClientSession` об'єктом, що відповідає так, як скаже тест.

### Приклад 3.1. `fake_session`: сесія, яка віддає наш HTML

`fake_session(html=..., error=...)` з `tests/unit/test_scraper.py`. `MagicMock` підтримує `async with`: його `__aenter__` — `AsyncMock`.

In [ ]:
import asyncio
from unittest import mock

import aiohttp

from news_hub import parser, scraper
from tests.unit.test_scraper import URL, fake_session, http_error

print(inspect.getsource(fake_session))
feed = html("rbc_newsline_item")

In [ ]:
page, items = await scraper.fetch_one(fake_session(feed), URL, t0=0.0)
print("новин:", page.count, "| помилка:", page.error)
print(items[0]["title"], "|", items[0]["datetime"])
assert (page.count, page.error) == (1, None)

`fetch_one` не знає, що сесія несправжня: він викликав `session.get(...)`, отримав «відповідь» і розібрав HTML.

### Приклад 3.2. Мок запам'ятовує виклики

Кожен виклик мока записується: з якими аргументами, скільки разів. Так тест перевіряє, що запит пішов з правильними заголовками (User-Agent):

In [ ]:
session = fake_session(feed)
await scraper.fetch_one(session, URL, t0=0.0)
args, kwargs = session.get.call_args
print("скільки викликів:", session.get.call_count)
print("адреса:", args[0])
print("User-Agent:", kwargs["headers"]["User-Agent"])
session.get.assert_called_once_with(URL, headers=scraper.HEADERS, timeout=mock.ANY)

`mock.ANY` дорівнює чому завгодно — тайм-аут нас тут не цікавить. Якби `fetch_one` забув `headers=`, `assert_called_once_with` упав би з описом різниці.

### Приклад 3.3. Мережа падає по-різному

Три помилки: сервер відповів `403`, тайм-аут, обрив з'єднання. Жодна не має вилетіти винятком на весь збір — кожна стає `page.error`.

In [ ]:
errors = [http_error(403), asyncio.TimeoutError(), aiohttp.ClientConnectionError("Connection reset by peer")]
messages = []
for error in errors:
    page, items = await scraper.fetch_one(fake_session(error=error), URL, t0=0.0)
    messages.append(page.error)
    print(f"{page.count} новин, error = {page.error!r}")
assert [m.split(":")[0] for m in messages] == ["ClientResponseError", "TimeoutError", "ClientConnectionError"]

`fake_session` кидає `403` з `raise_for_status` (сервер відповів), а тайм-аут і обрив — із самого `session.get` (відповіді не було). `fetch_one` ловить `aiohttp.ClientError` і `asyncio.TimeoutError` — і перетворює їх на рядок.

### Приклад 3.4. Одна функція — два імені

`scraper.py` робить `from .parser import parse_rbc_news`: у модулі `scraper` з'являється **власне** ім'я, що вказує на ту саму функцію. `mock.patch` міняє ім'я лише в **одному** модулі:

In [ ]:
print("одна функція, два імена:", scraper.parse_rbc_news is parser.parse_rbc_news)
with mock.patch("news_hub.parser.parse_rbc_news") as fake:
    print("patch news_hub.parser  → scraper бачить мок:", scraper.parse_rbc_news is fake)
with mock.patch("news_hub.scraper.parse_rbc_news") as fake:
    print("patch news_hub.scraper → scraper бачить мок:", scraper.parse_rbc_news is fake)
print("після with — знову справжня:", scraper.parse_rbc_news is parser.parse_rbc_news)

```text
до патча:                       news_hub.parser.parse_rbc_news ─┐
                                news_hub.scraper.parse_rbc_news ┴→ функція

patch("news_hub.parser…"):      news_hub.parser.parse_rbc_news  → Mock
                                news_hub.scraper.parse_rbc_news → функція   ← fetch_one шукає тут

patch("news_hub.scraper…"):     news_hub.parser.parse_rbc_news  → функція
                                news_hub.scraper.parse_rbc_news → Mock      ← fetch_one шукає тут
```

**🔮 Прогноз:** тест патчить `news_hub.parser.parse_rbc_news`, щоб парсер повернув порожній список. Що поверне `fetch_one` для сторінки з прикладу 3.1?

<details>
<summary>Відповідь</summary>

Одну новину — патч не подіяв. `fetch_one` шукає ім'я `parse_rbc_news` у модулі `scraper`, а патч замінив його лише в модулі `parser`. Правило **patch where used**: `news_hub.scraper.parse_rbc_news`.

</details>

### Задача 1. Patch where used

Наступна клітинка показує «неправильний» патч. Підміни парсер так, щоб `fetch_one` **справді** отримав порожній список:

- `mock.patch(<правильний шлях>, return_value=[])` як контекстний менеджер `with … as fake_parser:`;
- усередині — `await scraper.fetch_one(fake_session(feed), URL, t0=0.0)`, сторінку збережи в `page_patched`;
- після `with` перевір мок: `fake_parser.assert_called_once_with(feed)` — парсер отримав саме HTML сторінки.

In [ ]:
with mock.patch("news_hub.parser.parse_rbc_news", return_value=[]):
    wrong_page, _ = await scraper.fetch_one(fake_session(feed), URL, t0=0.0)
print("patch news_hub.parser — новин:", wrong_page.count)

In [ ]:
# YOUR CODE HERE


In [ ]:
assert page_patched.count == 0, "парсер не підмінився: патчити треба ім'я там, де його шукає fetch_one"
print("неправильний patch — новин:", wrong_page.count, "| правильний patch — новин:", page_patched.count)
assert scraper.parse_rbc_news is parser.parse_rbc_news, "після with має повернутися справжня функція"
print("✅ Задача 1 розв'язана")

### Приклад 3.5. Шпигун: `wraps=` — справжня функція під наглядом

Іноді не треба підміняти результат — лише перевірити, **чи** і **з чим** функцію викликали. `mock.patch(..., wraps=функція)` викликає справжню функцію і записує виклики:

In [ ]:
with mock.patch("news_hub.scraper.parse_rbc_news", wraps=parse_rbc_news) as spy:
    page, items = await scraper.fetch_one(fake_session(feed), URL, t0=0.0)
    await scraper.fetch_one(fake_session(error=http_error(403)), URL, t0=0.0)
print("новин:", page.count, "| викликів парсера:", spy.call_count)

Новина справжня (парсер працював), а виклик — один: на сторінці з `403` до парсера справа не дійшла. Такий тест ловить, наприклад, зайвий розбір сторінки з помилкою.

---
## 4. Фейк замість моку: локальний HTTP-сервер

Мок відповідає тим, що йому сказали. Код, що стоїть **між** запитом і рядком HTML, — з'єднання, заголовки, статус, **декодування байтів**, тайм-аут — під моком не виконується. **Фейк** — справжня, але спрощена реалізація: тут це HTTP-сервер на `127.0.0.1` з `aiohttp.test_utils.TestServer` на **вільному** порту.

### Приклад 4.1. Один битий байт

Сервер обіцяє UTF-8, а в кінці сторінки — байт `0xff`, якого в UTF-8 не буває. Визначимо дві сторінки — нормальну й биту:

In [ ]:
from urllib.parse import urlsplit

from aiohttp import web
from aiohttp.test_utils import TestServer

FEED_BYTES = (FIXTURES / "rbc_newsline_item.html").read_bytes()


async def good(request: web.Request) -> web.Response:
    return web.Response(body=FEED_BYTES, content_type="text/html", charset="utf-8")


async def broken(request: web.Request) -> web.Response:
    return web.Response(body=FEED_BYTES + b"<p>\xff</p>", content_type="text/html", charset="utf-8")


def make_site(**routes) -> web.Application:
    """web.Application з маршрутами /<ім'я>/ → обробник."""
    site = web.Application()
    for name, handler in routes.items():
        site.router.add_get(f"/{name}/", handler)
    return site

In [ ]:
async with TestServer(make_site(good=good, broken=broken)) as server:
    print("сервер:", server.make_url("/"))
    async with aiohttp.ClientSession() as session, session.get(server.make_url("/broken/")) as resp:
        try:
            await resp.text()                               # так читали сторінку раніше
        except UnicodeDecodeError as error:
            print(f"resp.text(): {type(error).__name__}; це aiohttp.ClientError? {isinstance(error, aiohttp.ClientError)}")
    outcome = await scraper.scrape_all_async([str(server.make_url(p)) for p in ("/broken/", "/good/")])
print("тепер:", [(urlsplit(p.url).path, p.count, p.error) for p in outcome.pages])

`UnicodeDecodeError` — **не** `aiohttp.ClientError`, тож `fetch_one` його не ловив: `asyncio.gather` передавав виняток нагору, і `POST /api/scrape` відповідав `500` — новини **решти** сторінок губилися через один байт. Тепер `fetch_one` читає `resp.text(errors="replace")`, і обидві сторінки розібрано. Під моком цієї помилки не видно в принципі: `response.text` там повертає готовий рядок, декодування немає.

### Приклад 4.2. Заголовок справді дійшов до сервера

Мок у прикладі 3.2 перевірив, що `fetch_one` **передав** `headers=`. Фейк перевіряє більше — що заголовок справді прийшов по мережі:

In [ ]:
seen_agents = []


async def recording(request: web.Request) -> web.Response:
    seen_agents.append(request.headers.get("User-Agent", ""))
    return web.Response(body=FEED_BYTES, content_type="text/html", charset="utf-8")

In [ ]:
async with TestServer(make_site(feed=recording)) as server:
    await scraper.scrape_all_async([str(server.make_url("/feed/"))])
print("сервер побачив:", seen_agents)
assert seen_agents == [scraper.HEADERS["User-Agent"]]

### Приклад 4.3. `gather` проти черги — тепер як тест

Дві «повільні» сторінки по 0,3 с. `scrape_all_async` бере їх одночасно, `scrape_sequential` — по черзі (урок 28):

In [ ]:
DELAY = 0.3


async def slow(request: web.Request) -> web.Response:
    await asyncio.sleep(DELAY)
    return web.Response(body=FEED_BYTES, content_type="text/html", charset="utf-8")

In [ ]:
async with TestServer(make_site(slow1=slow, slow2=slow)) as server:
    pages = [str(server.make_url("/slow1/")), str(server.make_url("/slow2/"))]
    together = await scraper.scrape_all_async(pages)
    one_by_one = await scraper.scrape_sequential(pages)
print(f"одночасно: {together.total_time:.2f} с | по черзі: {one_by_one.total_time:.2f} с")
assert together.total_time < 2 * DELAY <= one_by_one.total_time

Одночасно — трохи більше 0,3 с, по черзі — трохи більше 0,6 с. Мок відповідає миттєво, тож такий тест можливий лише з фейком.

**🔮 Прогноз:** додамо до сервера сторінку, що відповідає `403`, і зберемо дві сторінки: нормальну й заборонену. Скільки новин буде в `outcome.news`?

<details>
<summary>Відповідь</summary>

Одна — з нормальної сторінки. Сторінка з `403` стане `PageResult` з `error = "ClientResponseError: 403, …"` і нулем новин, а решта збору не постраждає.

</details>

### Задача 2. Сторінка з `403` на фейковому сервері

1. Напиши обробник `forbidden(request)`, що повертає `web.Response(status=403, text="<h1>403 Forbidden</h1>", content_type="text/html")`.
2. Підніми `TestServer(make_site(feed=good, forbidden=forbidden))` і збери `scrape_all_async` з двох адрес: `/feed/` і `/forbidden/` (у цьому порядку; адреси — `str(server.make_url(...))`).
3. Результат збору збережи в `outcome_403`.

In [ ]:
# YOUR CODE HERE


In [ ]:
assert len(outcome_403.pages) == 2, "збери дві сторінки: /feed/ і /forbidden/"
print([(urlsplit(p.url).path, p.count, (p.error or "")[:30]) for p in outcome_403.pages])
assert [p.count for p in outcome_403.pages] == [1, 0] and outcome_403.pages[1].error.startswith("ClientResponseError: 403")
assert len(outcome_403.news) == 1
print("✅ Задача 2 розв'язана")

### Приклад 4.4. Межа фейку: fakeredis і гонка

Фейк теж щось спрощує. `fakeredis` виконує команду одразу і **не віддає керування** циклу подій — тож одночасні корутини на ньому ніколи не перемежовуються. Порівняємо лічильник «прочитати → +1 → записати» (дві команди) і `INCR` (одна), по 10 одночасних викликів:

In [ ]:
import fakeredis
from redis.asyncio import Redis


async def naive_hit(redis: Redis, key: str) -> None:     # прочитати → +1 → записати
    count = int(await redis.get(key) or 0) + 1
    await redis.set(key, count)


async def race(redis: Redis) -> tuple[str, str]:
    await redis.delete("naive", "atomic")
    await asyncio.gather(*(naive_hit(redis, "naive") for _ in range(10)))
    await asyncio.gather(*(redis.incr("atomic") for _ in range(10)))
    result = (await redis.get("naive"), await redis.get("atomic"))
    await redis.aclose()
    return result

In [ ]:
print("fakeredis:    naive, atomic =", await race(fakeredis.FakeAsyncRedis(decode_responses=True)))
if os.getenv("TEST_REDIS_URL"):
    print("Redis-сервер: naive, atomic =", await race(Redis.from_url(os.environ["TEST_REDIS_URL"], decode_responses=True)))
else:
    print("Redis-сервер: пропущено — задай TEST_REDIS_URL=redis://localhost:6379/15, щоб порівняти")

На fakeredis «наївний» лічильник дорахував до 10 — гонки не видно. На справжньому Redis — далеко не 10 (щоразу по-різному): поки одна корутина чекала відповіді на `GET`, інші прочитали те саме значення. Тому атомарність rate limit (розділ 5) доводить **лише** прогін на справжньому Redis.

---
## 5. API асинхронно: `httpx.AsyncClient`

`TestClient` синхронний: застосунок крутиться в окремому потоці, запити йдуть по одному. `httpx.AsyncClient(transport=httpx.ASGITransport(app=app))` викликає FastAPI **напряму**, в тому ж циклі подій, що й ноутбук.

| | `TestClient` | `httpx.AsyncClient` + `ASGITransport` |
|---|---|---|
| `lifespan` | запускає сам | запускаєш сам: `app.router.lifespan_context(app)` |
| Redis / база в тесті | `client.portal.call(redis.get, …)` | `await redis.get(…)` |
| одночасні запити | ні, по одному | `asyncio.gather` |

### Приклад 5.1. `aclient` — як фікстура з `tests/integration/conftest.py`

`TestDatabase` з того самого `conftest.py` створює SQLite у пам'яті й підміняє залежності застосунку (`dependency_overrides`): базу, фабрику сесій і скрапери (заглушка `fake_scraper` з двома новинами). Зберемо з неї асинхронний контекстний менеджер — той самий порядок кроків, що у фікстурі:

In [ ]:
import logging
from contextlib import asynccontextmanager

import httpx

from news_hub.api import app
from news_hub.middleware import RATE_LIMIT_REQUESTS
from tests.integration.conftest import TestDatabase


@asynccontextmanager
async def aclient():
    """Тестова база + lifespan + чистий Redis + httpx.AsyncClient; після виходу — прибрати підміни."""
    db = TestDatabase()
    db.install()
    try:
        async with app.router.lifespan_context(app):          # ASGITransport lifespan не запускає
            await db.create_tables()
            await app.state.redis.flushdb()
            transport = httpx.ASGITransport(app=app)
            async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
                yield client
            await db.engine.dispose()
    finally:
        app.dependency_overrides.clear()

In [ ]:
async with aclient() as client:
    response = await client.get("/health")
    print(response.status_code, response.json(), "| X-Request-ID:", response.headers["X-Request-ID"][:8] + "…")
    print("підміни:", [dependency.__name__ for dependency in app.dependency_overrides])
print("після виходу підмін:", len(app.dependency_overrides))
logging.getLogger("news_hub").setLevel(logging.ERROR)        # далі журнал запитів не друкуємо

Рядок `INFO news_hub: … GET /health → 200` — журнал middleware `request_context` з уроку 40: id запиту, метод, шлях, статус, час. Далі в ноутбуці його вимкнено, щоб не заважав читати результати.

Усередині `async with` застосунок працює з тестовою базою; після виходу підміни прибрано — інакше вони «протекли» б у наступні приклади.

### Приклад 5.2. Кеш: `MISS`, потім `HIT` — і `await` прямо в Redis

Тест `test_cache_state_is_awaited_directly`: перший запит рахує відповідь (`MISS`), другий бере з Redis (`HIT`). Запис новин збільшує `news:version` — і старий кеш більше не читається.

In [ ]:
async with aclient() as client:
    print("1-й запит:", (await client.get("/api/news")).headers["X-Cache"])
    print("2-й запит:", (await client.get("/api/news")).headers["X-Cache"])
    print("версія кешу до збору:", await app.state.redis.get("news:version"))
    report = (await client.post("/api/scrape", json={"source": "live"})).json()
    print("збір:", {key: report[key] for key in ("news_found", "news_valid", "news_saved")}, "|", report["rejected"])
    print("версія кешу після:", await app.state.redis.get("news:version"))
    print("3-й запит:", (await client.get("/api/news")).headers["X-Cache"], [n["title"] for n in (await client.get("/api/news")).json()])

`source: "live"` тут не ходить в інтернет: `get_scrapers` підмінено заглушкою з двома новинами, і одну з них («Коротко») відхилила модель. Після запису версія кешу стала `1`, тож третій запит — знову `MISS` зі свіжими даними.

### Приклад 5.3. Rate limit рахує до перевірки тіла

Надішлемо **одночасно** (`asyncio.gather`) 10 запитів `POST /api/scrape` з неправильним тілом `{"source": "ftp"}`.

**🔮 Прогноз:** скільки відповідей буде `422` (неправильне тіло), а скільки `429` (rate limit)?

<details>
<summary>Відповідь</summary>

`RATE_LIMIT_REQUESTS` (5) відповідей `422` і решта `429`. Middleware rate limit стоїть **перед** ендпоінтом: він рахує запит до перевірки тіла. Інакше сервер можна було б засипати помилковими запитами без обмежень.

</details>

In [ ]:
async with aclient() as client:
    responses = await asyncio.gather(*(client.post("/api/scrape", json={"source": "ftp"}) for _ in range(10)))
statuses = sorted(r.status_code for r in responses)
print(statuses)
print("Retry-After:", next(r.headers.get("Retry-After") for r in responses if r.status_code == 429))
assert statuses == [422] * RATE_LIMIT_REQUESTS + [429] * (10 - RATE_LIMIT_REQUESTS)

**🔮 Прогноз:** якщо замінити `RateLimiter` на «наївний» (`GET`, +1, `SET`), чи впаде цей тест на `fakeredis`?

<details>
<summary>Відповідь</summary>

Ні — приклад 4.4: на `fakeredis` одночасні корутини не перемежовуються, гонки не буває. Атомарність доводить лише прогін на справжньому Redis (`TEST_REDIS_URL`) — там «наївний» лічильник пропускає зайві запити.

</details>

### Приклад 5.4. Перевірка тіла: 422 з поясненням

`ScrapeRequest` пропускає лише сторінки rbc.ua. Що скаже API, якщо попросити завантажити чужий сайт?

In [ ]:
async with aclient() as client:
    response = await client.post("/api/scrape", json={"pages": ["https://example.com/"]})
print(response.status_code)
print(response.json()["detail"][0]["msg"])

Сервер не завантажує довільні адреси на прохання будь-кого — інакше його можна змусити ходити туди, куди скаже зловмисник (SSRF, урок 47).

---
## 6. Покриття

`pytest-cov` показує, які рядки й гілки коду тести **виконали**. Налаштування — у `.coveragerc`:

In [ ]:
print("".join(line for line in open(".coveragerc", encoding="utf-8") if not line.startswith("# Урок")))

`branch = true` — рахувати ще й гілки (чи виконувались обидва виходи кожного `if`); `concurrency = thread,greenlet` — бо async SQLAlchemy виконує роботу з базою в greenlet, і без цього рядки сесії, `flush`, `commit` не потрапили б у звіт.

### Приклад 6.1. Покриття лише unit-тестами

Запустимо unit-тести з покриттям і знайдемо модуль з **найменшим** відсотком:

In [ ]:
def coverage_table(report: str) -> dict[str, int]:
    """Рядки звіту pytest-cov `news_hub/…` → {модуль: відсоток}."""
    rows = [line.split() for line in report.splitlines() if line.startswith("news_hub/")]
    return {row[0]: int(next(cell for cell in row if cell.endswith("%")).rstrip("%")) for row in rows}

In [ ]:
unit_report = run_pytest("-m", "unit", "--cov=news_hub", "--cov-report=term")
percent = coverage_table(unit_report)
for module, value in sorted(percent.items(), key=lambda pair: pair[1]):
    print(f"{module:25} {value:3}%")
lowest = min(percent, key=percent.get)
print("найменше:", lowest)

Угорі списку — код, що працює з базою чи Redis (`middleware.py`, `repository.py`, `cache.py`, `api.py`, `jobs.py`, `db.py`): unit-тести його й не мають зачіпати, його перевіряють інтеграційні. Покриття одного шару — не вирок, а карта: що перевіряє **який** шар.

### Приклад 6.2. Повний набір: лише непокрите

`term:skip-covered` ховає модулі зі 100%, а колонка `Missing` — номери рядків і гілок (`160->159` — гілка з рядка 160 на 159), яких не торкнувся жоден тест:

In [ ]:
full_report = run_pytest("--cov=news_hub", "--cov-report=term:skip-covered")
print("\n".join(line for line in full_report.splitlines()
                if line.startswith(("Name", "news_hub", "TOTAL")) or "skipped" in line or "passed" in line))

Лишилось кілька рядків — і їх свідомо не покрито: гілка справжнього Redis (її покриває прогін з `TEST_REDIS_URL`), перевірка типу для mypy, `__repr__`. **100% — не мета**; мета — щоб кожен червоний рядок був рішенням, а не випадком.

### Приклад 6.3. Справжній `get_db`, а не його копія

Тести API підміняють `get_db` копією з `conftest.py` — отже, самого `get_db` вони не виконують. Викличемо **оригінал** так, як це робить FastAPI: `anext(...)` — код до `yield` (сесія для ендпоінта); ще раз `anext` — код після `yield` (COMMIT); `athrow(виняток)` — так FastAPI передає помилку ендпоінта (ROLLBACK). Патчимо лише фабрику сесій, яку `get_db` шукає в модулі `db` (patch where used).

In [ ]:
from sqlalchemy.ext.asyncio import async_sessionmaker
from sqlalchemy.pool import StaticPool

from news_hub import db
from tests.integration.test_db import count, row


async def run_real_get_db(endpoint_fails: bool) -> tuple[str, int]:
    """Пройти get_db, як FastAPI; повертає (що сталося, скільки рядків у базі після)."""
    engine = db.make_engine("sqlite+aiosqlite://", poolclass=StaticPool)
    async with engine.begin() as conn:
        await conn.run_sync(db.Base.metadata.create_all)
    factory = async_sessionmaker(engine, expire_on_commit=False)
    with mock.patch.object(db, "SessionFactory", factory):
        dependency = db.get_db()
        session = await anext(dependency)              # код до yield
        session.add(row(1))
        await session.flush()                          # INSERT уже в базі, але в незавершеній транзакції
        try:
            if endpoint_fails:
                await dependency.athrow(RuntimeError("ендпоінт упав"))
            else:
                await anext(dependency)                # код після yield
        except StopAsyncIteration:
            outcome = "дійшли до кінця (COMMIT)"
        except RuntimeError as error:
            outcome = f"виняток пройшов далі: {error} (ROLLBACK)"
    rows = await count(factory)
    await engine.dispose()
    return outcome, rows

In [ ]:
for fails in (False, True):
    outcome, rows = await run_real_get_db(fails)
    print(f"ендпоінт упав: {fails!s:5} → {outcome}; рядків у базі: {rows}")

Успіх — рядок у базі; виняток — рядка немає, а виняток іде далі (FastAPI перетворить його на `500`). Саме це перевіряє `tests/integration/test_db.py`.

### Приклад 6.4. Тест ключа кешу

`NewsCache.key` будує ключ з параметрів запиту. Що має бути правдою?

1. Той самий набір параметрів в **іншому порядку** — той самий ключ: `?lang=uk&limit=5` і `?limit=5&lang=uk` — один запит.
2. Інші параметри — інший ключ.
3. Після `invalidate()` ключ змінюється — старий кеш більше не читається.

Шар — unit: потрібен лише Redis, і fakeredis тут достатньо (гонок у тесті немає). Тест приймає клас кешу параметром — щоб наступний приклад міг підставити зламаний.

In [ ]:
from news_hub.cache import NewsCache


async def check_cache_key(cache_class=NewsCache) -> None:
    cache = cache_class(fakeredis.FakeAsyncRedis(decode_responses=True))
    first = await cache.key("list", {"lang": "uk", "limit": 5})
    assert first == await cache.key("list", {"limit": 5, "lang": "uk"}), "порядок параметрів змінив ключ"
    assert first != await cache.key("list", {"lang": "ru", "limit": 5}), "інші параметри — той самий ключ"
    await cache.invalidate()
    assert first != await cache.key("list", {"lang": "uk", "limit": 5}), "invalidate не змінив ключ"

In [ ]:
await check_cache_key()
print("тест зелений:", await NewsCache(fakeredis.FakeAsyncRedis(decode_responses=True)).key("list", {"lang": "uk", "limit": 5}))

### Приклад 6.5. Чи ловить тест помилку?

Зелений тест нічого не доводить, якщо він зелений і на **зламаному** коді. Перевіримо: зробимо копію `NewsCache`, у якій ключ рахується без `sort_keys=True`.

In [ ]:
import hashlib
import json


class CacheWithoutSort(NewsCache):
    async def key(self, name, params):
        version = await self._redis.get(self.VERSION_KEY) or "0"
        digest = hashlib.sha1(json.dumps(params, ensure_ascii=False).encode()).hexdigest()[:12]   # без sort_keys
        return f"news:v{version}:{name}:{digest}"

In [ ]:
try:
    await check_cache_key(CacheWithoutSort)
    print("тест не помітив помилки!")
except AssertionError as error:
    print("тест упав, як і мав:", error)

Це ручна **мутаційна перевірка**: внесли помилку — тест має впасти. Так варто перевіряти кожен новий тест хоча б раз.

---
## 7. Межові значення

### Приклад 7.1. Знайди помилку: «майже rbc.ua»

Так перевіряли адресу в попередніх уроках — і тести були зелені:

In [ ]:
def is_rbc_host_old(host: str | None) -> bool:
    return (host or "").endswith("rbc.ua")

In [ ]:
for host in ("rbc.ua", "www.rbc.ua", "auto.rbc.ua"):
    assert is_rbc_host_old(host)
print("усі перевірки зелені")

Тести перевіряли лише те, що **має** пройти. Межових значень — «майже правильних» адрес — не було.

### Задача 3. Межові значення для перевірки адреси

1. Додай до словника `hosts` (адреса → чи має пройти) щонайменше дві адреси, які **не** мають пройти: такі, що схожі на rbc.ua, але не є ним чи його піддоменом. Хоч одна з них має обдурити `is_rbc_host_old`.
2. Напиши `is_rbc_host_fixed(host)`: `True` лише для `rbc.ua` або піддомену `*.rbc.ua`; для `None` і порожнього рядка — `False`.

In [ ]:
hosts = {"rbc.ua": True, "www.rbc.ua": True, "auto.rbc.ua": True}

In [ ]:
# YOUR CODE HERE


In [ ]:
missed = [host for host, allowed in hosts.items() if is_rbc_host_old(host) != allowed]
print("стара перевірка помиляється на:", missed)
assert missed, "додай до hosts адресу, на якій стара перевірка помиляється (схожу на rbc.ua)"
assert all(is_rbc_host_fixed(host) == allowed for host, allowed in hosts.items())
assert is_rbc_host_fixed(None) is False and is_rbc_host_fixed("news.rbc.ua") is True
print("✅ Задача 3 розв'язана")

### Приклад 7.2. Одна функція для двох місць

У проєкті правильна перевірка — `news_hub.models.is_rbc_host`, і нею користуються **обидва** місця: модель `NewsItem` (адреса новини) і `ScrapeRequest` (сторінки для збору). Перевіримо таблицею:

In [ ]:
from news_hub.api import ScrapeRequest
from news_hub.models import is_rbc_host

for host in ("rbc.ua", "www.rbc.ua", "auto.rbc.ua", "fakerbc.ua", "rbc.ua.evil.com"):
    try:
        ScrapeRequest(pages=[f"https://{host}/"])
        request_ok = "прийнято"
    except ValidationError:
        request_ok = "відхилено"
    print(f"{host:16} endswith('rbc.ua'): {host.endswith('rbc.ua')!s:5}  is_rbc_host: {is_rbc_host(host)!s:5}  ScrapeRequest: {request_ok}")

`fakerbc.ua` закінчується на `rbc.ua`, але відхилений: крапка перед `rbc.ua` — частина правила. Одне правило — в одній функції, щоб модель і API не розійшлися.

### Приклад 7.3. Мінімальні версії: `[None]` замість `[]`

`requirements.txt` обіцяє `beautifulsoup4>=4.12`. Прогін тестів на **найстаршій** дозволеній версії знайшов ще одну помилку: для тегу без атрибута `class` bs4 4.12 повертає `get_attribute_list("class") == [None]`, а новіші — `[]`. Відтворимо обидва варіанти без встановлення старої версії:

In [ ]:
def classes_old(attribute_list: list) -> str:
    return " ".join(attribute_list)


def classes_fixed(attribute_list: list) -> str:
    return " ".join(cls for cls in attribute_list if cls)

In [ ]:
for name, value in (("bs4 новіша", []), ("bs4 4.12", [None]), ("з класами", ["title", "big"])):
    try:
        old = repr(classes_old(value))
    except TypeError as error:
        old = f"TypeError: {error}"
    print(f"{name:11} {value!s:16} стара: {old:52} виправлена: {classes_fixed(value)!r}")

Тести на новій bs4 цього не бачили. Те, що обіцяє `requirements.txt` (мінімальні версії), теж треба перевіряти — у CI (урок 51) це стане окремим рядком матриці.

---
## Самоперевірка

1. Що перевіряє тест конвеєра, чого не перевіряють тести парсера й моделі окремо?
2. Де патчити `parse_rbc_news`, щоб підмінити його для `fetch_one`, і чому?
3. Яку помилку бачить фейковий сервер, але не бачить мок?
4. Чим `httpx.AsyncClient` + `ASGITransport` зручніший за `TestClient`, а в чому незручніший?
5. Чому 100% покриття — не мета?
6. Як переконатися, що новий тест справді ловить помилку?

<details>
<summary>Відповіді</summary>

1. Контракт між частинами: чи приймає модель саме те, що віддає парсер на справжній сторінці (приклади 2.4–2.5).
2. `news_hub.scraper.parse_rbc_news` — там, де ім'я **шукає** `fetch_one`; `from … import` створює в модулі `scraper` власне ім'я (приклад 3.4, задача 1).
3. Усе між запитом і рядком: декодування байтів, справжні заголовки, статуси, час (приклади 4.1–4.3).
4. Тест і застосунок в одному циклі подій: `await` до Redis напряму, `asyncio.gather` для одночасних запитів. Але `lifespan` і підміни треба запускати й прибирати самому (приклад 5.1).
5. Покриття каже лише, що рядок **виконався**, а не що його результат перевірено; деякі рядки свідомо лишають (гілка справжнього Redis). Мета — щоб кожен непокритий рядок був рішенням.
6. Внести в код помилку й побачити, що тест упав (приклад 6.5).

</details>

## Далі

- Книга курсу: [Урок 42](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_42/) — чотири помилки, які знайшли тести, покрокова схема «patch where used», межа фейку, покриття до/після.
- У папці `news_hub`: `pytest --cov=news_hub`; на справжніх серверах — `docker compose up -d db redis` і `TEST_DATABASE_URL=… TEST_REDIS_URL=redis://localhost:6379/15 pytest`.
- Урок 43 — AI-інструменти розробника: тести — перший спосіб перевірити згенерований код.